In [1]:
import tensorflow as tf
import tensorflow.keras as keras
import pickle
import functools

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from network import build_vggnet16
from train import WarmUpCosine, CustomWeightDecaySGD, LastNSaver, cifar_pad_crop_flip, mixup_batch, randaugment_mild, make_test_ds
from noise import noise_a, noise_p

In [4]:
num_classes = 3
initial_lr = 0.1
weight_decay = 1e-4
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [5]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [6]:
NN_16=[32,32,64,64,128,128,256,256]

In [7]:
model = build_vggnet16(NN_16,num_class=4)

In [8]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = CustomWeightDecaySGD(
    weight_decay=weight_decay,
    learning_rate=lr_schedule,
    momentum=0.9,
    nesterov=True
)

In [9]:
loss_fn=tf.keras.losses.CategoricalCrossentropy()
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy']
)

In [10]:
saver = LastNSaver(n=20)

In [11]:
gaussian_noise_batch = functools.partial(noise_a, a=0.2)
salt_pepper_noise_batch = functools.partial(noise_p, n=64)

In [12]:
def random_noise_batch(x, p_gaussian=0.5):
    x_g = gaussian_noise_batch(x)
    x_sp = salt_pepper_noise_batch(x)

    b = tf.shape(x)[0]
    use_g = tf.random.uniform([b, 1, 1, 1]) < p_gaussian
    use_g = tf.cast(use_g, x.dtype)

    return use_g * x_g + (1.0 - use_g) * x_sp


def clean_noisy_batch(x, y):
    x_noisy = random_noise_batch(x)

    x = tf.concat([x, x_noisy], axis=0)
    y = tf.concat([y, y], axis=0)
    return x, y

In [13]:
def make_train_ds(x_train, y_train_onehot, batch_size=128,
                  image_size=32, pad=4,
                  mixup_alpha=0.0,
                  strong_aug=False,
                  use_noise=True):

    ds = tf.data.Dataset.from_tensor_slices((x_train, y_train_onehot))
    ds = ds.shuffle(min(len(x_train), 20000), reshuffle_each_iteration=True)

    def to_float(x, y):
        x = tf.cast(x, tf.float32)
        y = tf.cast(y, tf.float32)
        return x, y

    ds = ds.map(to_float, num_parallel_calls=AUTOTUNE)

    ds = ds.map(
        lambda x, y: cifar_pad_crop_flip(x, y, image_size=image_size, pad=pad),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.map(
        lambda x, y: randaugment_mild(x, y, strong_aug=strong_aug),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(batch_size, drop_remainder=True)

    if use_noise:
        ds = ds.map(clean_noisy_batch, num_parallel_calls=AUTOTUNE)

    if mixup_alpha and mixup_alpha > 0:
        ds = ds.map(
            lambda xb, yb: mixup_batch(xb, yb, alpha=mixup_alpha),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.prefetch(AUTOTUNE)
    return ds

In [14]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = make_train_ds(
    x_train, y_train_onehot,
    batch_size=batch_size,
    image_size=32,
    pad=4,
    mixup_alpha=0.2,
    strong_aug=False,
    use_noise=True
)
test_ds = make_test_ds(x_test, y_test_onehot, batch_size=batch_size)

In [15]:
model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

Epoch 1/200
156/156 - 9s - loss: 1.2166 - accuracy: 0.4793 - val_loss: 1.5414 - val_accuracy: 0.2505 - 9s/epoch - 60ms/step
Epoch 2/200
156/156 - 2s - loss: 1.0043 - accuracy: 0.6049 - val_loss: 2.5343 - val_accuracy: 0.3842 - 2s/epoch - 16ms/step
Epoch 3/200
156/156 - 2s - loss: 0.8870 - accuracy: 0.6764 - val_loss: 0.7475 - val_accuracy: 0.7430 - 2s/epoch - 16ms/step
Epoch 4/200
156/156 - 2s - loss: 0.8144 - accuracy: 0.7211 - val_loss: 0.9273 - val_accuracy: 0.7007 - 2s/epoch - 16ms/step
Epoch 5/200
156/156 - 2s - loss: 0.7700 - accuracy: 0.7453 - val_loss: 0.8034 - val_accuracy: 0.7165 - 2s/epoch - 16ms/step
Epoch 6/200
156/156 - 2s - loss: 0.7348 - accuracy: 0.7607 - val_loss: 0.4800 - val_accuracy: 0.8305 - 2s/epoch - 16ms/step
Epoch 7/200
156/156 - 2s - loss: 0.6863 - accuracy: 0.7862 - val_loss: 0.4259 - val_accuracy: 0.8450 - 2s/epoch - 15ms/step
Epoch 8/200
156/156 - 2s - loss: 0.6802 - accuracy: 0.7901 - val_loss: 0.5395 - val_accuracy: 0.7940 - 2s/epoch - 16ms/step
Epoch 9/

In [16]:
model.save("VGG_11.h5")